### Create Predict Matchups

This notebook uses the information from the KenPom transformation of the predict dataset (current year) to build the match up table.  Each team needs to be matched up with every other team and the dataset should be aligned with the output from TransformKaggleData for training.

In [0]:
kenpom_df = spark.read.table("workspace.marchmadness.silver_kenpom_predict")
display(kenpom_df)

In [0]:
# Create all pairwise team matchups (full self join).
tourney_ids_df = kenpom_df.select('TeamID').distinct()

# Rename columns for cross-join
ids_1 = tourney_ids_df.withColumnRenamed('TeamID', 'Team1Id')
ids_2 = tourney_ids_df.withColumnRenamed('TeamID', 'Team2Id')

# Perform cross join to create all possible pairs
matchups_df = ids_1.crossJoin(ids_2)

# Optionally: Remove same-team pairs
matchups_df = matchups_df.filter(matchups_df.Team1Id != matchups_df.Team2Id).orderBy(['Team1Id', 'Team2Id'])

display(matchups_df)

In [0]:
# Find the current year from kenpom_df.Year as a scalar
current_year = kenpom_df.select("Year").orderBy("Year", ascending=False).first()["Year"]

# Add new columns to matchups_df
from pyspark.sql.functions import concat_ws, lit

matchups_df = matchups_df.withColumn(
    "Team1Season_Team",
    concat_ws("_", lit(current_year), matchups_df.Team1Id)
).withColumn(
    "Team2Season_Team",
    concat_ws("_", lit(current_year), matchups_df.Team2Id)
).withColumn(
    "id",
    concat_ws("_", lit(current_year), matchups_df.Team1Id, matchups_df.Team2Id)
)

matchups_df = matchups_df.select('id', 'Team1Id', 'Team2Id', 'Team1Season_Team', 'Team2Season_Team')
display(matchups_df)

In [0]:
# Save the transformed KenPom data to the mode-specific silver table
matchups_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable('marchmadness.silver_predict_matchups')